[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/04_overfitting_and_bias_variance.ipynb)

# Bài 04 — Underfitting, Overfitting và Bias–Variance Tradeoff

**🎯 Mục tiêu**
- Nhận diện underfitting và overfitting qua lỗi train/validation.
- Hiểu **bias** và **variance** bằng mô phỏng — không chỉ bằng định nghĩa.
- Phân rã lỗi: $\text{lỗi} = \text{bias}^2 + \text{variance} + \text{nhiễu}$.
- Các cách chống overfitting / underfitting; regularization như một "núm vặn" độ phức tạp.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Underfitting và Overfitting

<img src="images/overfitting_1.png" style="width:70%;height:70%;">

- **Underfitting (high bias):** model **quá đơn giản**, không nắm được quy luật → lỗi cao trên **cả** train lẫn validation.
- **Overfitting (high variance):** model **quá phức tạp**, học cả **nhiễu** của dữ liệu train → lỗi train rất thấp nhưng validation cao.
- **Vừa phải:** lỗi train và validation đều thấp, gần nhau.

Với bài toán phân loại, overfitting thể hiện ở **decision boundary** uốn éo bám theo từng điểm:

<img src="images/overfitting_2.png" style="width:70%;height:70%;">

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary

X, y = make_classification_2d("moons", n=300, noise=0.3)
X_val, y_val = make_classification_2d("moons", n=1000, noise=0.3, seed=1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (k, title) in zip(axes, [(150, "k = 150: underfit"), (15, "k = 15: vừa phải"), (1, "k = 1: overfit")]):
    model = KNeighborsClassifier(k).fit(X, y)
    plot_decision_boundary(model, X, y, ax=ax)
    ax.set_title(f"{title}\nacc train = {model.score(X, y):.2f} | acc validation = {model.score(X_val, y_val):.2f}")
plt.show()

## 2. Bias và Variance — nhìn bằng mô phỏng

Tưởng tượng ta có thể lấy **nhiều tập train khác nhau** từ cùng một quy luật, fit model trên mỗi tập:

- **Bias:** dự đoán **trung bình** (qua các tập train) lệch khỏi hàm thật bao nhiêu → sai số **hệ thống**, do model quá đơn giản.
- **Variance:** dự đoán **dao động** bao nhiêu giữa các tập train → model nhạy cảm với dữ liệu cụ thể.

<img src="images/Bias-Variance.png" style="width:45%;height:45%;">

In [ ]:
def true_f(x):
    return np.sin(1.5 * x)


def poly(d):
    return make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())


grid = np.linspace(-2.5, 2.5, 200)   # đánh giá ở vùng bên trong (tránh ngoại suy ở 2 mép)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, d in zip(axes, [1, 5, 13]):
    preds = []
    for _ in range(40):                                  # 40 tập train khác nhau, mỗi tập 40 điểm
        x = rng.uniform(-3, 3, 40)
        yy = true_f(x) + rng.normal(0, 0.3, 40)
        p = poly(d).fit(x[:, None], yy).predict(grid[:, None])
        preds.append(p)
        ax.plot(grid, p, color="#93c5fd", lw=0.8)
    preds = np.array(preds)
    ax.plot(grid, preds.mean(0), "b", lw=2.5, label="dự đoán trung bình")
    ax.plot(grid, true_f(grid), "k--", lw=2, label="hàm thật")
    bias2 = np.mean((preds.mean(0) - true_f(grid)) ** 2)
    var = np.mean(preds.var(0))
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"bậc {d}: bias² = {bias2:.3f} | variance = {var:.3f}")
    ax.legend(fontsize=8, loc="lower right")
plt.show()

- Bậc 1: các đường gần như **giống nhau** (variance thấp) nhưng **đều sai** (bias cao).
- Bậc 5: vừa phải — bias và variance đều nhỏ.
- Bậc 13: từng đường **dao động dữ dội** (variance cao) dù có tới 40 điểm mỗi tập.

## 3. Phân rã lỗi (bias–variance decomposition)

Với dữ liệu $y = f(x) + \varepsilon$, $\operatorname{Var}(\varepsilon) = \sigma^2$, lỗi bình phương kỳ vọng tại một điểm $x$ (lấy kỳ vọng qua các tập train khác nhau):

$$\mathbb E\big[(y - \hat f(x))^2\big] = \underbrace{\big(\mathbb E[\hat f(x)] - f(x)\big)^2}_{\text{bias}^2} + \underbrace{\mathbb E\big[(\hat f(x) - \mathbb E\hat f(x))^2\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{nhiễu không thể giảm}}$$

Tăng độ phức tạp model → bias giảm, variance tăng. Lỗi tổng nhỏ nhất ở độ phức tạp **vừa phải** — đó là **bias–variance tradeoff**:

In [ ]:
degrees = range(1, 14)
bias2_list, var_list = [], []
for d in degrees:
    preds = []
    for _ in range(60):
        x = rng.uniform(-3, 3, 40)
        yy = true_f(x) + rng.normal(0, 0.3, 40)
        preds.append(poly(d).fit(x[:, None], yy).predict(grid[:, None]))
    preds = np.array(preds)
    bias2_list.append(np.mean((preds.mean(0) - true_f(grid)) ** 2))
    var_list.append(np.mean(preds.var(0)))
bias2_arr, var_arr = np.array(bias2_list), np.array(var_list)

fig, ax = plt.subplots()
ax.plot(degrees, bias2_arr, "o-", label="bias²")
ax.plot(degrees, var_arr, "s-", label="variance")
ax.plot(degrees, bias2_arr + var_arr + 0.3**2, "k^-", label="lỗi tổng = bias² + variance + nhiễu")
ax.axhline(0.3**2, color="gray", ls=":", label="nhiễu σ² (không thể giảm)")
ax.set_yscale("log")
ax.set_xlabel("bậc đa thức")
ax.legend()
ax.set_title("Bias–variance tradeoff")
plt.show()

> 💡 Deep learning hiện đại cho thấy với model **cực lớn** (nhiều tham số hơn nhiều so với số mẫu) lỗi có thể giảm trở lại (**double descent**). Nhưng với hầu hết bài toán classical ML, bức tranh chữ U ở trên vẫn là kim chỉ nam.

## 4. Khắc phục

| Vấn đề | Dấu hiệu | Cách khắc phục |
|---|---|---|
| **High bias (underfit)** | lỗi train cao, train ≈ validation | model phức tạp hơn; thêm feature / feature phi tuyến; giảm regularization; train lâu hơn |
| **High variance (overfit)** | lỗi train thấp, validation cao hơn nhiều | **thêm dữ liệu**; tăng regularization; giảm số feature; model đơn giản hơn; ensemble (bagging); early stopping; data augmentation |

Bài 05 dùng **learning curve** để chẩn đoán chính xác hơn nên thử cách nào.

## 5. Regularization — một "núm vặn" độ phức tạp

Thay vì giảm bậc đa thức, giữ model phức tạp nhưng **phạt trọng số lớn** (chương 02 bài 07 — MAP; chương 06 — Ridge/Lasso):

$$J(\mathbf w) = \text{MSE}(\mathbf w) + \lambda\lVert\mathbf w\rVert_2^2$$

$\lambda$ lớn → model "đơn giản" hơn (bias ↑, variance ↓).

In [ ]:
x = rng.uniform(-3, 3, 25)
yy = true_f(x) + rng.normal(0, 0.3, 25)
grid = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, lam in zip(axes, [1e-6, 0.1, 100]):
    m = make_pipeline(PolynomialFeatures(12), StandardScaler(), Ridge(alpha=lam)).fit(x[:, None], yy)
    ax.scatter(x, yy, s=20)
    ax.plot(grid, true_f(grid), "k--", lw=1)
    ax.plot(grid, m.predict(grid[:, None]), "r", lw=2)
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"đa thức bậc 12, λ = {lam:g}")
plt.show()

Cùng một model bậc 12: λ rất nhỏ → overfit; λ vừa → khớp tốt; λ rất lớn → underfit. **Chọn λ bằng validation** (chương 06, 15, 16).

## 🧠 Kiểm tra nhanh

1. Accuracy train 99%, validation 72%. Bias hay variance cao? Nêu 3 cách khắc phục.
2. Accuracy train 70%, validation 69% với bài toán con người làm được 95%. Vấn đề là gì?
3. Vì sao thêm dữ liệu giúp giảm variance nhưng gần như không giảm bias?
4. Tăng $\lambda$ trong Ridge ảnh hưởng thế nào tới bias và variance?

## 📌 Tóm tắt

- Underfit = bias cao (model quá đơn giản); overfit = variance cao (model học cả nhiễu).
- Lỗi = bias² + variance + nhiễu; độ phức tạp tối ưu ở giữa.
- Chống overfit: thêm dữ liệu, regularization, model đơn giản hơn, ensemble, early stopping.
- Chống underfit: model mạnh hơn, thêm feature, giảm regularization.

## 📚 Tài liệu tham khảo

- Andrew Ng — CS229 lecture notes, *Bias-Variance and Error Analysis*
- Scott Fortmann-Roe — *Understanding the Bias-Variance Tradeoff* (2012)
- Belkin et al. — *Reconciling modern machine learning practice and the bias-variance trade-off* (PNAS 2019) — double descent
- Hình minh hoạ overfitting/bias–variance: từ Tuần 4 khoá [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning) (MIT License)